In [2]:
#Install spaCy
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


I installed spaCy to build a small extractive summarizer. The model will select important sentences from my text instead of writing new ones. This keeps the project simple and avoids a long training process.

In [4]:
#Import libraries
import spacy

from collections import Counter
from heapq import nlargest

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

print("spaCy is ready.")

spaCy is ready.


I created a basic English pipeline and added sentence detection. This lets spaCy divide the text into sentences and individual words.

In [5]:
corpus = """
Harry Potter grows up with the Dursleys, unaware that he belongs to
the wizarding world. Hagrid visits Harry and explains that he is a
wizard who has been invited to attend Hogwarts. Harry also learns
that his parents were killed by Voldemort and that he survived
Voldemort's attack as a baby.

Harry travels to Hogwarts and becomes friends with Ron Weasley
and Hermione Granger. The Sorting Hat places Harry in Gryffindor.
At Hogwarts, Harry begins learning magic and discovers that school
life includes both exciting opportunities and difficult challenges.
His friendship with Ron and Hermione becomes increasingly important
throughout the school year.

Harry discovers that a valuable magical object is being protected
at Hogwarts. The object is the Philosopher's Stone, which can produce
an elixir that extends life. Harry and his friends suspect that
someone is trying to steal the Stone. They begin looking for clues
about the Stone and the protections surrounding it.

Harry initially suspects Professor Snape because of his unfriendly
behavior and several events that seem suspicious. However, Harry's
suspicions do not reveal the full situation. The mystery surrounding
the Stone shows that appearances can lead people to incorrect
conclusions.

Harry, Ron, and Hermione decide to investigate the threat to the Stone.
Each friend contributes different skills as they face the obstacles
protecting it. Hermione uses her knowledge and reasoning, while Ron
helps the group through a dangerous chess challenge. Their teamwork
allows Harry to continue toward the final chamber.

Harry discovers that Professor Quirrell is trying to obtain the Stone
for Voldemort. Voldemort has been using Quirrell to pursue a return
to power. Harry prevents Quirrell from taking the Stone and survives
their confrontation. Dumbledore later explains important parts of
the events to Harry.

The Stone is destroyed to prevent it from being used by Voldemort.
Harry finishes his first year with a stronger connection to the
wizarding world. His experiences at Hogwarts show the importance
of friendship, courage, and looking beyond first impressions.
"""

doc = nlp(corpus)

print("Number of sentences:", len(list(doc.sents)))
print("Original word count:", len(corpus.split()))

Number of sentences: 26
Original word count: 328


I wanted to use the plot about Harry Potter's first year at Hogwarts from the book. I kept the topic focused on the mystery of the
Philosopher's Stone so the corpus would be specific and manageable. The text includes background information, friendships, and the main conflict for the summarizer to compare.

In [7]:
#Text normalizer
def normalize_tokens(document):
    return [
        token.lower_
        for token in document
        if token.is_alpha and not token.is_stop
    ]


normalized_words = normalize_tokens(doc)

print("First 30 normalized words:")
print(normalized_words[:30])

First 30 normalized words:
['harry', 'potter', 'grows', 'dursleys', 'unaware', 'belongs', 'wizarding', 'world', 'hagrid', 'visits', 'harry', 'explains', 'wizard', 'invited', 'attend', 'hogwarts', 'harry', 'learns', 'parents', 'killed', 'voldemort', 'survived', 'voldemort', 'attack', 'baby', 'harry', 'travels', 'hogwarts', 'friends', 'ron']


I created a text normalization function to prepare words for scoring. It changes words to lowercase and removes stop words, punctuation, and numbers.

In [9]:
#Build Summarization model
class HarryPotterSummarizer:
    def __init__(self, nlp):
        self.nlp = nlp

    def summarize(self, text, number_of_sentences=5):
        if number_of_sentences < 1:
            raise ValueError(
                "Choose at least one summary sentence."
            )

        document = self.nlp(text)

        #Normalize the words used for scoring
        words = normalize_tokens(document)

        if not words:
            return "", []

        #Count each word's appearances
        word_counts = Counter(words)

        #Convert counts into relative weights
        highest_count = max(word_counts.values())

        word_weights = {
            word: count / highest_count
            for word, count in word_counts.items()
        }

        sentence_scores = []

        for position, sentence in enumerate(document.sents):
            sentence_words = normalize_tokens(sentence)

            if not sentence_words:
                continue

            #Average word weights
            score = sum(
                word_weights[word]
                for word in sentence_words
            ) / len(sentence_words)

            sentence_scores.append({
                "position": position,
                "sentence": " ".join(sentence.text.split()),
                "score": score
            })

        #Select the highest scoring sentences
        selected = nlargest(
            min(number_of_sentences, len(sentence_scores)),
            sentence_scores,
            key=lambda item: item["score"]
        )

        #Return selected sentences in their original order
        selected.sort(key=lambda item: item["position"])

        summary = " ".join(
            item["sentence"] for item in selected
        )

        return summary, sentence_scores

I built a summarizer that gives more weight to frequently used words. Each sentence receives the average weight of its normalized words. The model selects the highest-scoring sentences and puts them back in their original order. Averaging helps prevent long sentences from winning just because they contain more words.

In [10]:
#Generate summary
summarizer = HarryPotterSummarizer(nlp)

summary, sentence_scores = summarizer.summarize(
    corpus,
    number_of_sentences=5
)

print("Harry Potter Summary\n")
print(summary)

original_count = len(corpus.split())
summary_count = len(summary.split())

print("\nOriginal word count:", original_count)
print("Summary word count:", summary_count)

if original_count:
    reduction = (
        1 - summary_count / original_count
    ) * 100

    print(f"Text reduced by: {reduction:.1f}%")

Harry Potter Summary

Harry and his friends suspect that someone is trying to steal the Stone. However, Harry's suspicions do not reveal the full situation. Harry, Ron, and Hermione decide to investigate the threat to the Stone. Harry discovers that Professor Quirrell is trying to obtain the Stone for Voldemort. Harry prevents Quirrell from taking the Stone and survives their confrontation.

Original word count: 328
Summary word count: 58
Text reduced by: 82.3%


I asked the model to select five sentences from the Harry Potter corpus. I printed the summary and compared its word count with the original to see how much shorter it became. The model selected sentences based on word frequency, so a shorter result does not automatically mean it captured all the important events. Which here, one might not understand what the stone is or what is happening. I still need to check whether the selected sentences explain the main conflict, make sense together, and avoid repeating information.

In [11]:
#Review sentence scores
ranked_sentences = sorted(
    sentence_scores,
    key=lambda item: item["score"],
    reverse=True
)

print("Top 5 Sentences\n")

for rank, item in enumerate(ranked_sentences[:5], start=1):
    print(f"{rank}. Score: {item['score']:.3f}")
    print(item["sentence"])
    print()

Top 5 Sentences

1. Score: 0.312
Harry and his friends suspect that someone is trying to steal the Stone.

2. Score: 0.312
Harry, Ron, and Hermione decide to investigate the threat to the Stone.

3. Score: 0.312
Harry discovers that Professor Quirrell is trying to obtain the Stone for Voldemort.

4. Score: 0.297
However, Harry's suspicions do not reveal the full situation.

5. Score: 0.277
Harry prevents Quirrell from taking the Stone and survives their confrontation.



I checked the highest-scoring sentences to see why they were selected. Repeated names and words can strongly influence the results. For example, sentences mentioning Harry or the Stone may receive higher scores, even if another sentence explains an important event. The scores measure word frequency rather than full understanding.

In [13]:
#Compare summary length
for sentence_count in [3, 5]:
    comparison_summary, _ = summarizer.summarize(
        corpus,
        number_of_sentences=sentence_count
    )

    print(f"\n{sentence_count}-Sentence Summary\n")
    print(comparison_summary)
    print("\nWord count:", len(comparison_summary.split()))


3-Sentence Summary

Harry and his friends suspect that someone is trying to steal the Stone. Harry, Ron, and Hermione decide to investigate the threat to the Stone. Harry discovers that Professor Quirrell is trying to obtain the Stone for Voldemort.

Word count: 38

5-Sentence Summary

Harry and his friends suspect that someone is trying to steal the Stone. However, Harry's suspicions do not reveal the full situation. Harry, Ron, and Hermione decide to investigate the threat to the Stone. Harry discovers that Professor Quirrell is trying to obtain the Stone for Voldemort. Harry prevents Quirrell from taking the Stone and survives their confrontation.

Word count: 58


I compared 3-sentence and 5-sentence summaries to see how the length affects the information included in the summary. A shorter summary is quicker to read, but it may leave out background or important context. A longer summary provides more detail, although it can still miss useful information.

I created a summarization model using spaCy and an original overview of Harry Potter and the sorceror's stone. I normalized the words, calculated word frequencies, and ranked sentences to create shorter summaries. This project helped me understand that selecting sentences is different from understanding a story. The model uses repeated words as clues, so it might leave out an important event or select sentences that repeat similar information. I would review the output for accuracy, coverage, and readability before deciding whether it is a useful summary.

References

Rowling, J. K. (1997). Harry Potter and the sorcerer’s stone (Vol. 1). Scholastic Inc.